## VLM Shellgame multiview

- VLM is **gemini-3.8-flash (processing mode: 'agentic')** (model performed very well in single view shellgame)
→ A,B,C labeled on table, visible in cam1 (POV). At start, show where puck is, move camera to other view cam1, shuffle cups, move back into cam0 view and place on A B and C markers.
- 3 types of videos:
  - shellgame_mv_a – cam0 and cam1 look in same direction
  - shellgame_mv_b – cam0 and cam1 across from eachother (lateral inversion)
  - shellgame_mv_c – cam0 same and cam1 on some diagonal angle or even left or right view camera.
- For each video model picks from A,B,C


## Result:



In [1]:
## conda env vgd has openai
import base64
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import numpy as np
from tqdm import tqdm

# import cv2

In [2]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

# Preparing main index_df with vqa videos

In [3]:
## index_df
index_df = pd.read_csv('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/index.csv')
print(index_df.shape)

index_df = index_df[index_df['session'].str.startswith('shellgame_mv')].reset_index(drop=True)
print(index_df.shape)

index_df['cam'] = 'cam0'
index_df = index_df.rename(columns={'id':'video_id'})

print(index_df.shape)
index_df.head(2)

(285, 4)
(41, 4)
(41, 5)


,video_id,template,placeholders,session,cam
0,261125-104457,C,PUCK,shellgame_mv_a,cam0
1,261125-104554,B,PUCK,shellgame_mv_a,cam0


In [4]:
labels_list = index_df.template.unique().tolist()
labels_list.sort()

index_df.template.value_counts()

template
C    18
A    16
B     7
Name: count, dtype: int64

In [5]:
index_df.session.value_counts()

session
shellgame_mv_a    18
shellgame_mv_c    12
shellgame_mv_b    11
Name: count, dtype: int64

### select videos

In [6]:
for i, row in index_df.iterrows():
    print(f"'{row['video_id']}', # --> {row['template']:5} -- {row['session']} ")


'261125-104457', # --> C     -- shellgame_mv_a 
'261125-104554', # --> B     -- shellgame_mv_a 
'261125-104705', # --> C     -- shellgame_mv_a 
'261125-104732', # --> A     -- shellgame_mv_a 
'261125-104759', # --> A     -- shellgame_mv_a 
'261125-104837', # --> C     -- shellgame_mv_a 
'261125-104917', # --> A     -- shellgame_mv_a 
'261125-104952', # --> A     -- shellgame_mv_a 
'261125-105943', # --> C     -- shellgame_mv_a 
'261125-110012', # --> C     -- shellgame_mv_a 
'261125-110038', # --> A     -- shellgame_mv_a 
'261125-110106', # --> A     -- shellgame_mv_a 
'261125-110135', # --> C     -- shellgame_mv_a 
'261125-110711', # --> A     -- shellgame_mv_a 
'261125-110736', # --> C     -- shellgame_mv_a 
'261125-110758', # --> C     -- shellgame_mv_a 
'261125-110830', # --> A     -- shellgame_mv_a 
'261125-110901', # --> B     -- shellgame_mv_a 
'261125-111934', # --> A     -- shellgame_mv_b 
'261125-112001', # --> C     -- shellgame_mv_b 
'261125-112026', # --> A     -- shellgam

In [7]:
vids_list = [
    '261125-104457', # --> C     -- shellgame_mv_a 
    '261125-104554', # --> B     -- shellgame_mv_a 
    '261125-104705', # --> C     -- shellgame_mv_a 
    '261125-104732', # --> A     -- shellgame_mv_a 
    '261125-104759', # --> A     -- shellgame_mv_a 
    '261125-104837', # --> C     -- shellgame_mv_a 
    '261125-104917', # --> A     -- shellgame_mv_a 
    '261125-104952', # --> A     -- shellgame_mv_a 
    '261125-105943', # --> C     -- shellgame_mv_a 
    '261125-110012', # --> C     -- shellgame_mv_a 
    '261125-110038', # --> A     -- shellgame_mv_a 
    '261125-110106', # --> A     -- shellgame_mv_a 
    '261125-110135', # --> C     -- shellgame_mv_a 
    '261125-110711', # --> A     -- shellgame_mv_a 
    '261125-110736', # --> C     -- shellgame_mv_a 
    '261125-110758', # --> C     -- shellgame_mv_a 
    '261125-110830', # --> A     -- shellgame_mv_a 
    '261125-110901', # --> B     -- shellgame_mv_a 
    '261125-111934', # --> A     -- shellgame_mv_b 
    '261125-112001', # --> C     -- shellgame_mv_b 
    '261125-112026', # --> A     -- shellgame_mv_b 
    '261125-112050', # --> B     -- shellgame_mv_b 
    '261125-112116', # --> C     -- shellgame_mv_b 
    '261125-112139', # --> A     -- shellgame_mv_b 
    '261125-112205', # --> B     -- shellgame_mv_b 
    '261125-112306', # --> C     -- shellgame_mv_b 
    '261125-112333', # --> A     -- shellgame_mv_b 
    '261125-112521', # --> C     -- shellgame_mv_b 
    '261125-112540', # --> C     -- shellgame_mv_b 
    '261125-114107', # --> C     -- shellgame_mv_c 
    '261125-114146', # --> A     -- shellgame_mv_c 
    '261125-114209', # --> C     -- shellgame_mv_c 
    '261125-114232', # --> B     -- shellgame_mv_c 
    '261125-114306', # --> C     -- shellgame_mv_c 
    '261125-114327', # --> B     -- shellgame_mv_c 
    '261125-114354', # --> C     -- shellgame_mv_c 
    '261125-114427', # --> A     -- shellgame_mv_c 
    '261125-114451', # --> A     -- shellgame_mv_c 
    '261125-114516', # --> C     -- shellgame_mv_c 
    '261125-114540', # --> B     -- shellgame_mv_c 
    '261125-114606', # --> A     -- shellgame_mv_c 
]
print(len(vids_list))

41


In [8]:
print(index_df.shape)
index_df = index_df[index_df.video_id.isin(vids_list)]
print(index_df.shape)

(41, 5)
(41, 5)


In [13]:
import shutil
# ------------- chunk to copy videos ------------- #
source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

# 2. Clear destination directory if it exists, then recreate it
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
os.makedirs(dest_dir, exist_ok=True)

# 3. Get unique video IDs and copy matching files
unique_video_ids = vids_list

for filename in os.listdir(source_dir):
    if any(video_id in filename for video_id in unique_video_ids):
        shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
# ------------------------------------------------ #

# VLM Evaluation 

### Compress Videos. 9second videos too long and high def. api call can handle <10MB vids and mine are well over 15mb per single view

#### 1. compress to lower res and store as mp4 (both cam0 and cam1)
#### 2. Concatenate the 2 views per video carefully - Because API cannot ingest 2 separate videos at the same time --> combined video for multiview evaluation

In [ ]:
import os
import subprocess
from tqdm import tqdm

raw_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw"
compressed_dir = "data/compressed_videos"

os.makedirs(compressed_dir, exist_ok=True)

pbar = tqdm(vids_list, desc="Compressing Videos")
for vid in pbar:
    # Strip extension if vids_list already includes '.avi'
    stem = os.path.splitext(vid)[0]
    
    input_path = os.path.join(raw_dir, f"{stem}_cam1.avi")       # Change to f"{stem}_cam0.avi" if suffix is still used
    output_path = os.path.join(compressed_dir, f"{stem}_cam1.mp4")

    if os.path.exists(output_path):
        print(f"[-] Skipping (already exists): {stem}_cam1.mp4")
        continue

    if not os.path.exists(input_path):
        print(f"[!] Warning (missing input): {input_path}")
        continue

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-loglevel", "error",
            "-i", input_path,
            "-vf", "fps=20,scale=-2:480p",
            "-c:v", "libx264",
            "-crf", "23",
            "-an",
            output_path,
        ],
        check=True,
    )

    orig_mb = os.path.getsize(input_path) / (1024 * 1024)
    new_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"[*] Converted {stem}: {orig_mb:.2f} MB -> {new_mb:.2f} MB")

Compressing Videos:   0%|          | 0/41 [00:00<?, ?it/s]

In [ ]:
## 31mins total to compress all 41 cam0 videos to 480p 20fps mp4 format.
## now doing same videos cam1 with a tqdm progressbar.

In [ ]:
def concatenate_videos_side_by_side(video_path_0, video_path_1, output_path):
    """Stitches two synchronized videos horizontally into a single video file."""
    if os.path.exists(output_path):
        print(f"[-] Skipping concatenation: '{output_path}' already exists.")
        return
    
    cap0 = cv2.VideoCapture(video_path_0)
    cap1 = cv2.VideoCapture(video_path_1)

    fps = cap0.get(cv2.CAP_PROP_FPS) or 30.0
    width0 = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
    height0 = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
    height1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Match height of video 1 to video 0 if they differ
    target_height = height0

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    # Combined width = width of cam0 + width of cam1
    out_width = width0 + int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH) * (target_height / height1))
    out = cv2.VideoWriter(output_path, fourcc, fps, (out_width, target_height))

    while cap0.isOpened() and cap1.isOpened():
        ret0, frame0 = cap0.read()
        ret1, frame1 = cap1.read()

        if not ret0 or not ret1:
            break

        # Resize frame1 if heights don't match exactly
        if frame1.shape[0] != target_height:
            new_w = int(frame1.shape[1] * (target_height / frame1.shape[0]))
            frame1 = cv2.resize(frame1, (new_w, target_height))

        # Concatenate horizontally (Left: View 0 | Right: View 1)
        combined_frame = cv2.hconcat([frame0, frame1])
        out.write(combined_frame)

    cap0.release()
    cap1.release()
    out.release()

    print(f"[*] New file created: {output_path}")

In [ ]:
videos_file_path = "data/compressed_videos/"
combined_videos_file_path = 'data/combined_videos/'

for video_name in vids_list:
    path_cam0 = f"{videos_file_path}{video_name}_cam0.mp4"
    path_cam1 = f"{videos_file_path}{video_name}_cam1.mp4"
    combined_path = f"{combined_videos_file_path}{video_name}_combined.mp4"

    # Stitch the videos horizontally
    concatenate_videos_side_by_side(path_cam0, path_cam1, combined_path)

### Run one video

In [9]:
# Load environment variables from .env
load_dotenv()

True

In [10]:
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"), # openrouter change
    base_url="https://openrouter.ai/api/v1", # openrouter change
    )
client

In [ ]:
# Encode combined video to base64
video_name = vids_list[0]

video_file_path = f'data/combined_videos/{video_name}_combined.mp4'

with open(video_file_path, "rb") as f:
    b64_video = base64.b64encode(f.read()).decode("utf-8")
    b64_size_mb = len(b64_video) / (1024 ** 2)
    print(f"Base64 payload size: {b64_size_mb:.2f} MB")

Base64 payload size: 2.69 MB


In [12]:
len(b64_video)

2816428

In [13]:
assert not b64_video.startswith("data:"), "Duplicate data: prefix detected"
assert "\n" not in b64_video, "Newlines found in base64 string"

In [ ]:
system_prompt = (
    "You are an expert multi-view visual object-tracking and spatial-temporal reasoning model. "
    "You are provided with a single synchronized split-screen video showing two side-by-side camera views of the exact same event: "
    "the LEFT half is 'cam0' and the RIGHT half is 'cam1'. "
    "In 'cam1', there are three clearly marked positions: A, B, and C, each starting with an inverted cup. "
    "At the start of the video, a puck is revealed under one of the cups in 'cam1'. "
    "The cups are then moved out of 'cam1' into 'cam0', shuffled inside 'cam0', and finally returned to 'cam1' onto markers A, B, and C. "
    "You must synthesize spatial and temporal cues across both camera views to continuously maintain the identity of the target cup across camera boundaries and determine its final resting position."
)

user_prompt = """
Analyze this synchronized split-screen video ('cam0' on the left, 'cam1' on the right) of a multi-view shell game and determine which position (A, B, or C) in 'cam1' hides the puck at the end of the video.

Valid Positions: {"A", "B", "C"}

Instructions:
1. Initial Reveal ('cam1' - Right): Identify the starting position (A, B, or C) where the puck is initially revealed under a cup at the beginning of the recording.
2. Track the cup with the puck chronologically:
  - Outbound Transition ('cam1' -> 'cam0'): Track the target cup as it leaves 'cam1' and enters 'cam0', noting its relative spatial order (e.g., left, middle, right) upon entering 'cam0'. 
  - Shuffle Tracking ('cam0' - Left): Continuously track the physical target cup through every swap and movement inside 'cam0'. 
  - Return Transition ('cam0' -> 'cam1'): Track the target cup as it exits 'cam0', re-enters 'cam1', and is placed onto one of the final markers (A, B, or C).
3. State your final predicted position ("A", "B", or "C") where the puck is located at the end of the video.
4. Provide a confidence score as a float between 0.0 and 1.0 denoting your certainty in the final predicted position (where 1.0 is completely certain and 0.33 is random guessing).
5. State the exact number of frames you sampled and processed from the given video in integer format.
6. Spatial Camera Setup Inference: Provide a single, concise sentence describing your best inference of the physical camera setup and the spatial position/orientation of 'cam0' relative to 'cam1' (e.g., adjacent adjacent top-down views, angled side-by-side cameras, or opposite-facing perspectives) based on how the cups and hands exit one frame and enter the other.

Output strictly in JSON matching this schema:
{
  "initial_position": "<A, B, or C>",
  "swap_sequence": "cam1:A -> enter_cam0:left -> cam0_swap:right -> return_cam1:B",
  "predicted_position": "<A, B, or C>",
  "confidence": 0.0,
  "frames_seen": 60,
  "camera_setup_reasoning": "<One sentence describing the inferred physical position and orientation of cam0 relative to cam1>"
}
"""
print(user_prompt)

In [15]:
len(system_prompt.split(' ')) + len(user_prompt.split(' '))

229

In [ ]:
start_time = time.time()

response = client.chat.completions.create(
    model="google/gemini-3.8-flash",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    # "text": "Whats happening in this video? please describe scene and activity.",
                    "text": user_prompt,
                },
                {
                    "type": "video_url",
                    "video_url": {
                        "url": f"data:video/mp4;base64,{b64_video}",
                        "processing": "agentic",
                        },
                },
            ],
        },
    ],
    extra_body={
        "reasoning": {"effort": "medium"},
    },
    response_format={"type": "json_object"},
    temperature=0.1,

)

print("Prompt tokens:", response.usage.prompt_tokens)

elapsed_time = round(time.time() - start_time, 2)

print(f"Elapsed time: {elapsed_time}s")

Prompt tokens: 85110
Elapsed time: 108.84s


In [17]:
prediction_json = json.loads(response.choices[0].message.content)
prediction_json.keys()

dict_keys(['initial_position', 'swap_sequence', 'predicted_position', 'confidence'])

In [18]:
response.usage

CompletionUsage(completion_tokens=10166, prompt_tokens=85110, total_tokens=95276, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=9722, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=76994, image_tokens=None, text_tokens=None, video_tokens=0), cost=0.04998405, is_byok=False, cost_details={'upstream_inference_cost': 0.04998405, 'upstream_inference_prompt_cost': 0.01186155, 'upstream_inference_completions_cost': 0.0381225})

In [ ]:
# Extract JSON prediction content
prediction_json = json.loads(response.choices[0].message.content)

# Extract token & performance metadata
usage = response.usage

# 3. Update the dictionary to log both video file names

## this try except is from my testing when i wasnt giving the user_prompt to the models and model was returning confidence like 'low','high'
# try: 
#     conf = float(prediction_json.get("confidence", 0.0))
# except ValueError:
#     conf = 0.0

run_result = {
    "video_file": os.path.basename(video_file_path),
    "model": response.model,
    "elapsed_seconds": elapsed_time,
    "prompt_tokens": usage.prompt_tokens if usage else None,
    "completion_tokens": usage.completion_tokens if usage else None,
    "total_tokens": usage.total_tokens if usage else None,
    "cost": usage.cost if usage else None,
    "initial_position": prediction_json.get("initial_position"),
    "predicted_position": prediction_json.get("predicted_position"),
    "confidence": float(prediction_json.get("confidence", 0.0)),
    # "confidence": conf,
    "swap_sequence": prediction_json.get("swap_sequence"),
    "frames_seen": prediction_json.get("frames_seen"),
}

# Append result as a single JSON line
with open('data/vlm_eval_results/shellgame_predictions_testing_models.jsonl', "a", encoding="utf-8") as f:
    f.write(json.dumps(run_result) + "\n")

print(json.dumps(run_result, indent=2))

{
  "video_file": "261117-155127_cam0.mp4",
  "model": "google/gemini-3.8-flash",
  "elapsed_seconds": 108.84,
  "prompt_tokens": 85110,
  "completion_tokens": 10166,
  "total_tokens": 95276,
  "cost": 0.04998405,
  "initial_position": "A",
  "predicted_position": "C",
  "confidence": 0.98,
  "swap_sequence": "A -> B -> C -> A -> B -> C"
}


In [20]:
# 1. Load jsonl file directly into a DataFrame
df_testing_models = pd.read_json("data/vlm_eval_results/shellgame_predictions_testing_models.jsonl", lines=True)
pd.set_option('display.float_format', '{:,.3f}'.format)
print(df_testing_models.shape)
print(df_testing_models)
print(f"\nTotal Cost of Testing: $ {round(sum(df_testing_models.cost), 3)}")

(18, 11)
                video_file                          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens  cost initial_position predicted_position  confidence               swap_sequence
0   261117-155127_cam0.mp4                   z-ai/glm-5.3           14.070            123               1859          1982 0.008              NaN                NaN       0.000                         NaN
1   261117-155127_cam0.mp4             z-ai/glm-5.3-flash            1.810            123                120           243 0.001              NaN                NaN       0.330                         NaN
2   261117-155127_cam0.mp4  google/gemini-3-flash-preview            2.440            113                 90           203 0.000              NaN                NaN       0.000                         NaN
3   261117-155127_cam0.mp4  google/gemini-3-flash-preview            2.340            878                 50           928 0.001                A                  A       

## Main Experiment: For loop over many videos

In [ ]:
# Load environment variables from .env
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"), # openrouter change
    base_url="https://openrouter.ai/api/v1", # openrouter change
    )

In [79]:
len(vids_list[1:])

31

In [ ]:
from tqdm import tqdm

videos_dir = 'data/compressed_videos/'
results_path = "data/vlm_eval_results/shellgame_predictions_gemini-3.8-flash-agentic.jsonl"

pbar = tqdm(vids_list, desc="Processing Videos")

for video_name in pbar:
    # Make file path and load video in base64 format
    video_file_path = videos_dir + video_name + '_cam0.mp4'
    with open(video_file_path, "rb") as f:
        b64_video = base64.b64encode(f.read()).decode("utf-8")

    # system_prompts and user_prompt exactly same for all videos.
    
    try:
        # API call
        start_time = time.time()

        response = client.chat.completions.create(
            model="google/gemini-3.8-flash",
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": user_prompt,
                        },
                        {
                            "type": "video_url",
                            "video_url": {
                                "url": f"data:video/mp4;base64,{b64_video}",
                                "processing": "agentic",
                                },
                        },
                    ],
                },
            ],
            extra_body={
                # "provider": {
                #     "order": ["Google AI Studio"],  
                #     "allow_fallbacks": False,
                # },
                "reasoning": {"effort": "medium"},
            },
            response_format={"type": "json_object"},
            temperature=0.1,
            # timeout=180,
        )

        elapsed_time = round(time.time() - start_time, 2)

        # Extract JSON prediction content
        prediction_json = json.loads(response.choices[0].message.content)

        # Extract token & performance metadata
        usage = response.usage

        # 3. Update the dictionary to log both video file names
        run_result = {
            "video_file": os.path.basename(video_file_path),
            "model": response.model,
            "elapsed_seconds": elapsed_time,
            "prompt_tokens": usage.prompt_tokens if usage else None,
            "completion_tokens": usage.completion_tokens if usage else None,
            "total_tokens": usage.total_tokens if usage else None,
            "cost": usage.cost if usage else None,
            "initial_position": prediction_json.get("initial_position"),
            "predicted_position": prediction_json.get("predicted_position"),
            "confidence": float(prediction_json.get("confidence", 0.0)),
            "swap_sequence": prediction_json.get("swap_sequence"),
        }

        # Append result as a single JSON line
        with open(results_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(run_result) + "\n")

    except Exception as e:
        print(f"[!] Error processing video {video_name}: {e}")
        continue



Processing Videos: 100%|██████████| 26/26 [07:53<00:00, 18.20s/it]


# Loading and analyzing the jsonl outputs of the VLM
## Main Experiment: shellgame 
### gemini-3.8-flash (processing:'static')

In [9]:
# 1. Load jsonl file directly into a DataFrame
df = pd.read_json("data/vlm_eval_results/shellgame_predictions_gemini-3.8-flash.jsonl", lines=True)

print(df.shape)
print(df.head(10))

(32, 11)
               video_file                    model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens      cost initial_position predicted_position  confidence swap_sequence
0  261117-155127_cam0.mp4  google/gemini-3.8-flash            34.07            882               3715          4597  1.46e-02                A                  B        0.95        A -> B
1  261117-155207_cam0.mp4  google/gemini-3.8-flash           131.87            882              14540         15422  5.52e-02                C                  C        0.95   C -> B -> C
2  261117-155252_cam0.mp4  google/gemini-3.8-flash           175.50            948              19004         19952  7.20e-02                A                  C        0.95   A -> B -> C
3  261117-155426_cam0.mp4  google/gemini-3.8-flash            84.66            948               9023          9971  3.45e-02                B                  C        0.85        B -> C
4  261117-155503_cam0.mp4  google/gemini-3.8-flash 

In [10]:
index_df['video_file'] = index_df.video_id + '_cam0.mp4'
print(index_df.shape)

df = pd.merge(df, index_df, how = 'left', on = ['video_file']) # LEFT JOIN
print(df.shape)
print(df.sample(3))

(32, 6)
(32, 16)
                video_file                    model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens  cost initial_position predicted_position  confidence swap_sequence       video_id template placeholders           session   cam
3   261117-155426_cam0.mp4  google/gemini-3.8-flash            84.66            948               9023          9971  0.03                B                  C        0.85        B -> C  261117-155426        C         PUCK  shellgame-medium  cam0
24  261117-162259_cam0.mp4  google/gemini-3.8-flash            38.96            816               4286          5102  0.02                C                  C        0.95        C -> C  261117-162259        A         PUCK    shellgame-slow  cam0
12  261117-161053_cam0.mp4  google/gemini-3.8-flash            53.19            816               5971          6787  0.02                C                  C        0.95   C -> B -> C  261117-161053        B         PUCK    shellgame-fast  cam0

In [11]:
df['correct_top1'] = df['predicted_position'] == df['template']

print(f'Examples: {df.shape[0]}')
print(f'correct: {df.correct_top1.sum():4}')
print(f'top1acc: {df.correct_top1.mean():4}')

Examples: 32
correct:   17
top1acc: 0.53125


In [10]:
import textwrap
import pandas as pd

pd.set_option('display.float_format', '{:,.3f}'.format)

def analyze_df(df):
    print('=' * 20, 'DF Stats by Session', '=' * 20)

    stats = df.groupby(['session']).agg(
        videos=('video_file', 'count'),
        top1_acc=('correct_top1', 'mean'),
        avg_confidence=('confidence', 'mean'),
        avg_prompt_tokens=('prompt_tokens', 'mean'),
        avg_completion_tokens=('completion_tokens', 'mean'),
        total_cost_usd=('cost', 'sum'),
        avg_frames_seen=('frames_seen', 'mean'),
        sd_frames_seen=('frames_seen', 'std')
    )
    print(stats)
    print('\n')

    print('=' * 30, 'Individual Videos by session', '=' * 30)

    for session in df['session'].unique():
        print('\n' + '=' * 30, session, '=' * 30)

        for i, row in df[df['session'] == session].iterrows():
            # Include ground-truth column if present in your merged DataFrame
            summary_cols = [
                col for col in [
                    'video_id', 'session', 'initial_position',
                    'template', 'predicted_position', 'confidence', 'correct_top1'
                ] if col in df.columns
            ]
            print(row[summary_cols].to_frame().T.to_string(index=False))

            print('\nSwap Sequence: ', textwrap.fill(str(row['swap_sequence']), width=100))
            print('-' * 60)

In [13]:
analyze_df(df)

==================== DF Stats by Session ====================
                  videos  top1_acc  avg_confidence  avg_prompt_tokens  avg_completion_tokens  total_cost_usd
session                                                                                                     
shellgame-fast        10     0.300           0.957            875.400              5,856.300           0.226
shellgame-medium      10     0.500           0.930            921.600             12,873.100           0.490
shellgame-slow        12     0.750           0.955            898.500              4,151.750           0.195


============================== Individual Videos by session ==============================

============================== shellgame-medium ==============================
     video_id          session initial_position template predicted_position confidence correct_top1
261117-155127 shellgame-medium                A        C                  B      0.950        False

Swap Sequence:  A -

### A little random pred experiment

In [26]:
from random import shuffle

x = [i for i in range(10)]
print(x)
shuffle(x)
print(x)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 8, 7, 2, 3, 9, 5, 4, 0, 6]


In [24]:
import numpy as np

rng = np.random.default_rng(49)
df["random_pred"] = rng.choice(["A", "B", "C"], size=len(df))

df['correct_top1_rand'] = df['random_pred'] == df['template']

print(f'Examples: {df.shape[0]}')
print(f'correct: {df.correct_top1_rand.sum():4}')
print(f'top1acc: {df.correct_top1_rand.mean():4}')

Examples: 30
correct:    9
top1acc:  0.3


In [ ]:
df.groupby(['session']).agg(
        videos=('video_file', 'count'),
        top1_acc=('correct_top1_rand', 'mean'),
        avg_confidence=('confidence', 'mean'),
)

,videos,top1_acc,avg_confidence
session,,,
shellgame-fast,8,0.375,0.999
shellgame-medium,10,0.000,0.963
shellgame-slow,12,0.500,0.958


In [28]:
set(vids_list) - set(df.video_id)

{'261117-155543', '261117-160858', '261117-160936', '261117-161121'}

### gemini-3.8-flash (processing:'agentic')

In [11]:
# 1. Load jsonl file directly into a DataFrame
df = pd.read_json("data/vlm_eval_results/shellgame_predictions_gemini-3.8-flash-agentic.jsonl", lines=True)

print(df.shape)
print(df.head(2))

(32, 12)
               video_file                    model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens  cost initial_position predicted_position  confidence                    swap_sequence  frames_seen
0  261117-155127_cam0.mp4  google/gemini-3.8-flash          169.250          62000              18051         80051 0.080                A                  C       0.980  A -> B -> C -> B -> A -> B -> C          152
1  261117-155207_cam0.mp4  google/gemini-3.8-flash           56.850          10832               5660         16492 0.022                C                  B       1.000                 C -> B -> A -> B           65


In [12]:
index_df['video_file'] = index_df.video_id + '_cam0.mp4'
print(index_df.shape)

df = pd.merge(df, index_df, how = 'left', on = ['video_file']) # LEFT JOIN
print(df.shape)
print(df.sample(3))

(32, 6)
(32, 17)
                video_file                    model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens  cost initial_position predicted_position  confidence     swap_sequence  frames_seen       video_id template placeholders           session   cam
21  261117-161859_cam0.mp4  google/gemini-3.8-flash           69.700          16971               6614         23585 0.030                C                  A       1.000       C -> B -> A           97  261117-161859        A         PUCK    shellgame-slow  cam0
18  261117-161455_cam0.mp4  google/gemini-3.8-flash          118.380          39371              11517         50888 0.053                A                  B       0.980  A -> B -> C -> B          138  261117-161455        B         PUCK  shellgame-medium  cam0
15  261117-161257_cam0.mp4  google/gemini-3.8-flash          132.190          61740              13838         75578 0.064                C                  B       0.950       C -> A -> B      

In [13]:
df['correct_top1'] = df['predicted_position'] == df['template']

print(f'Examples: {df.shape[0]}')
print(f'correct: {df.correct_top1.sum():4}')
print(f'top1acc: {df.correct_top1.mean():4}')

Examples: 32
correct:   25
top1acc: 0.78125


In [19]:
set(vids_list) - set(df.video_id)

{'261117-155621', '261117-160858', '261117-161017', '261117-161150'}

In [14]:
analyze_df(df)

==================== DF Stats by Session ====================
                  videos  top1_acc  avg_confidence  avg_prompt_tokens  avg_completion_tokens  total_cost_usd  avg_frames_seen  sd_frames_seen
session                                                                                                                                      
shellgame-fast        10     0.400           0.970         56,782.000             15,918.300           0.696          133.800          68.083
shellgame-medium      10     1.000           0.976         57,537.100             13,579.000           0.618          131.000          56.224
shellgame-slow        12     0.917           0.990         14,798.750              5,193.333           0.277           75.583          13.859


============================== Individual Videos by session ==============================

============================== shellgame-medium ==============================
     video_id          session initial_position templat